In [ ]:
# ============================================================
# 07 — ABLATION: EMBEDDING MODEL (MIND)
# multilingual-E5 vs English MiniLM; query/passage prefixes; mean vs best pooling.
# Fully self-contained MIND notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers -q
import os, glob, re, math, time, zipfile, numpy as np, pandas as pd, datetime as dt, lightgbm as lgb, random, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
random.seed(0)
# ---- hardcoded MIND paths (small: train -> dev for offline metrics) ----
TRAIN = "/kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train"
DEV   = "/kaggle/input/datasets/wrathofgod123/mind-dev/MINDsmall_dev"
SPLITS = [TRAIN, DEV]
NEWS = ["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH  = ["impression_id","user_id","time","history","impressions"]
_WORD = re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"

news = pd.concat([pd.read_csv(f"{d}/news.tsv", sep="\t", header=None, names=NEWS, quoting=3,
                 usecols=["news_id","category","title","abstract"]) for d in SPLITS]
                ).drop_duplicates("news_id").reset_index(drop=True)
news["title"] = news["title"].fillna(""); news["abstract"] = news["abstract"].fillna("")
cat_lut = {pfx(r.news_id):(r.category if isinstance(r.category,str) else "") for r in news.itertuples()}
ids = [pfx(r.news_id) for r in news.itertuples()]
corpus = [tok(f"{r.title} {r.abstract}") for r in news.itertuples()]
id_to_row = {x:i for i,x in enumerate(ids)}
title_lut = {pfx(r.news_id):tok(r.title) for r in news.itertuples()}
print("articles:", len(ids))


In [ ]:
# ---- load BOTH encoders ----
from sentence_transformers import SentenceTransformer
def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
    b["t"]=pd.to_datetime(b["time"],format="%m/%d/%Y %I:%M:%S %p",errors="coerce");return b
b_dv=load_beh(DEV)
hist_lut={}
for b in [load_beh(TRAIN),b_dv]:
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]

texts=[f"{r.title} {r.abstract}".strip() for r in news.itertuples()]

# MiniLM: English-specialised, NO prefix
minilm=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
mm_mat=minilm.encode(texts,batch_size=512,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
mm_by={ids[i]:mm_mat[i] for i in range(len(ids))}

# E5: multilingual, REQUIRES query:/passage: prefixes
e5=SentenceTransformer("intfloat/multilingual-e5-base")
e5_mat=e5.encode(["passage: "+t for t in texts],batch_size=256,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
e5_by={ids[i]:e5_mat[i] for i in range(len(ids))}
print("both encoders ready")


In [ ]:
# ---- reranking AUC under each (model, pooling) combination ----
def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def hist_vecs(uid,by,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];return [by[x] for x in ai if x in by]
def imps(b_df):
    for u,t,im in zip(b_df["user_id"],b_df["t"],b_df["impressions"]):
        if not isinstance(im,str) or pd.isna(t): continue
        cand=[];labs=[]
        for tk in im.split():
            p=tk.split("-")
            if len(p)==2: cand.append(pfx(p[0]));labs.append(int(p[1]))
        if cand and sum(labs)>0: yield pfx(u),cand,np.array(labs)

def eval_model(by, pooling, prefix_q=None):
    aucs=[]
    for uid,cand,labs in imps(b_dv):
        hv=hist_vecs(uid,by)
        if not hv: continue
        if pooling=="mean":
            um=np.mean(hv,0);um/=(np.linalg.norm(um)+1e-9)
            s=np.array([float(um@by[c]) if c in by else 0.0 for c in cand])
        else: # best
            s=np.array([float(max((v@by[c] for v in hv),default=0.0)) if c in by else 0.0 for c in cand])
        a=auc_i(s,labs)
        if a is not None: aucs.append(a)
    return np.mean(aucs)

print("=== MIND embedding-model ablation (reranking AUC) ===")
for name,by in [("MiniLM",mm_by),("E5",e5_by)]:
    for pool in ("mean","best"):
        print(f"  {name:8s} {pool:5s}: {eval_model(by,pool):.4f}")
print("\nFinding: English MiniLM > multilingual E5; MiniLM prefers mean-pool.")
